In [5]:
import requests
import pandas as pd
import time

# URL base
url = "https://www.drogariasaopaulo.com.br/api/catalog_system/pub/products/search"

# Lista de SKUs
sku_list = [
"629138","628425","627526","578525","567337","548014","542725","529346",
"515167","511102","510890","387975","360201","295710","290220","288292",
"222011","219789","204005","195480","183130","165760","68209","51853",
"907081","906476","906425","898392","896888","896870","893552","886513",
"883530","883166","883131","875198","870331","868744","864668","862401",
"861162","861111","860816","860174","855499","855308","852414","837113",
"497398","476455","474193","473855","465100","430838","423866","385948",
"385034","360325","288284","271683","222470","208949","203548","194247",
"183377","165514","145971","145807","134597","108618","108588","95249",
"89010","87300","80900","73113","41858","41580","33804","28916","27537",
"906450","906433","906417","901490","900753","897205","897060","897051",
"896896","896470","896233","896195","896152","895970","895962"
]

todos = []

# Loop por SKU individual
for i, sku in enumerate(sku_list, 1):
    params = {
        "_from": 0,
        "_to": 49,  # só precisamos de 1 produto por vez
        "fq": f"skuId:{sku}"  # apenas 1 SKU por requisição
    }
    
    try:
        response = requests.get(url, params=params, timeout=(10,30))
        response.raise_for_status()
        data = response.json()
        
        if data:  # se a API retornou algum produto
            todos.extend(data)
        
        print(f"SKU {i}/{len(sku_list)} coletado: {sku} → Total produtos: {len(todos)}")
        time.sleep(0.3)  # evita bloqueio do site
    
    except requests.HTTPError as e:
        print(f"Erro no SKU {sku}: {e}")
        time.sleep(1)  # espera um pouco e continua

# Transformar em DataFrame completo
df = pd.DataFrame(todos)

# Criar DataFrame somente com campos relevantes
df_limpo = pd.DataFrame([
    {
        "id": produto.get("productId"),
        "nome": produto.get("productName"),
        "sku": produto.get("items")[0]["itemId"] if produto.get("items") else None,
        "preco": produto.get("items")[0]["sellers"][0]["commertialOffer"]["Price"] 
                 if produto.get("items") else None
    }
    for produto in todos
])

sku_retornados = [produto["items"][0]["itemId"] for produto in todos if produto.get("items")]
skus_nao_encontrados = [sku for sku in sku_list if sku not in sku_retornados]

print("SKUs não encontrados / ausentes na API:")
print(skus_nao_encontrados)

# Salvar em CSV
df_limpo.to_csv("produtos_drogariasp.csv", index=False)
print("CSV salvo com sucesso")

SKU 1/96 coletado: 629138 → Total produtos: 1
SKU 2/96 coletado: 628425 → Total produtos: 2
SKU 3/96 coletado: 627526 → Total produtos: 3
SKU 4/96 coletado: 578525 → Total produtos: 4
SKU 5/96 coletado: 567337 → Total produtos: 5
SKU 6/96 coletado: 548014 → Total produtos: 6
SKU 7/96 coletado: 542725 → Total produtos: 7
SKU 8/96 coletado: 529346 → Total produtos: 8
SKU 9/96 coletado: 515167 → Total produtos: 9
SKU 10/96 coletado: 511102 → Total produtos: 10
SKU 11/96 coletado: 510890 → Total produtos: 11
SKU 12/96 coletado: 387975 → Total produtos: 12
SKU 13/96 coletado: 360201 → Total produtos: 13
SKU 14/96 coletado: 295710 → Total produtos: 14
SKU 15/96 coletado: 290220 → Total produtos: 15
SKU 16/96 coletado: 288292 → Total produtos: 16
SKU 17/96 coletado: 222011 → Total produtos: 17
SKU 18/96 coletado: 219789 → Total produtos: 18
SKU 19/96 coletado: 204005 → Total produtos: 19
SKU 20/96 coletado: 195480 → Total produtos: 20
SKU 21/96 coletado: 183130 → Total produtos: 21
SKU 22/96 

In [6]:
display(df_limpo)


,id,nome,sku,preco
0,629138,Kit Aparelho De Barbear Descartavel Gillette P...,629138,24.49
1,628425,Lubrificante Íntimo K-Med Original 100g,628425,25.99
2,627526,Kit Loção Hidratante Nivea Milk Pele Seca a Ex...,627526,40.59
3,578525,Sabonete Líquido de Glicerina Johnson's Baby 4...,578525,37.99
4,567337,Fio Dental Oral-B Expert Limpeza Profunda 2 Un...,567337,39.19
...,...,...,...,...
91,896233,Creme Dental Original Mint Colgate Total Preve...,896233,6.99
92,896195,Creme Dental Original Mint Colgate Total Preve...,896195,38.19
93,896152,Kit Protetor Solar Isdin Fusion Water FPS60 50...,896152,124.80
94,895970,Kit Protetor Solar FPS 70 Nivea Sun Protect & ...,895970,62.90
